In [11]:
import sys
sys.path.append('../')

import pandas as pd
from functools import partial
from data_clients.polymarket_client import PolymarketAPIClient
from trading_rules import entries as e, exits as ex, engine
from trading_rules.metrics import Metrics
from trading_rules.fees import FeeParams, FeeModel
from trading_rules.fills import SlippageParams
from util.data_processor import TickDataIntervalEnum


In [12]:
# FETCH THE DATA STEP AND PROCESS IT
MARKET_SLUG = 'will-jesus-christ-return-in-2025'
client = PolymarketAPIClient()
market = client.get_price_history_by_outcome(MARKET_SLUG, desired_outcome="No", interval=TickDataIntervalEnum.FIVE_MINUTES)
market["symbol"] = MARKET_SLUG
resampled_df = market.resample("10min").last()
resampled_df.ffill()

Will Jesus Christ return in 2025?
Not Will Jesus Christ return in 2025?
requested start and end: 2024-12-31 12:00:00+00:00 2025-12-31 12:00:00+00:00


,timestamp,open,high,low,close,volume,outcome_id,symbol
timestamp_formatted,,,,,,,,
2025-03-20 22:10:00+00:00,1.742509e+12,0.9700,0.9700,0.9700,0.9700,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:20:00+00:00,1.742510e+12,0.9695,0.9695,0.9695,0.9695,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:30:00+00:00,1.742510e+12,0.9650,0.9650,0.9650,0.9650,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:40:00+00:00,1.742511e+12,0.9670,0.9670,0.9670,0.9670,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:50:00+00:00,1.742511e+12,0.9610,0.9610,0.9610,0.9610,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
...,...,...,...,...,...,...,...,...
2025-12-31 11:10:00+00:00,1.767180e+12,0.9995,0.9995,0.9995,0.9995,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-12-31 11:20:00+00:00,1.767180e+12,0.9995,0.9995,0.9995,0.9995,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-12-31 11:30:00+00:00,1.767181e+12,0.9995,0.9995,0.9995,0.9995,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025


In [13]:
resampled_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 41123 entries, 2025-03-20 22:10:00+00:00 to 2025-12-31 11:50:00+00:00
Freq: 10min
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   timestamp   41100 non-null  float64
 1   open        41100 non-null  float64
 2   high        41100 non-null  float64
 3   low         41100 non-null  float64
 4   close       41100 non-null  float64
 5   volume      41100 non-null  float64
 6   outcome_id  41100 non-null  object 
 7   symbol      41100 non-null  object 
dtypes: float64(6), object(2)
memory usage: 2.8+ MB


In [14]:
# RUN THE SIMULATED BACKTEST
DAYS5_BARS = 720
entries_partials = [
    partial(e.mean_reversion, data=resampled_df, window=DAYS5_BARS)
]
entries_series = [entry(data=resampled_df) for entry in entries_partials]
# AND all the entry rules
entries = pd.concat(entries_series, axis=1).all(axis=1)
exit_rules = [ex.time_exit(max_bars=144)]

fee_param = FeeParams(
    model = FeeModel.POLYMARKET,
    rate_bps=500,
    rebate_share=0.0,
    min_fee_dollars=0.0
)

slippage_params = SlippageParams(
   bps = 2.0,
   atr_multiplier = 0.1
)

result = engine.backtest(
    data=resampled_df,
    entries_series=entries,
    exits_list=exit_rules,
    initial_cash=10000,
    fee_params=fee_param,
    slippage_params=slippage_params
)



In [15]:
# MEASURE PERFORMANCE USING METRICS
metrics = Metrics(result['trades'], result['equity'])
metrics.summary(periods_per_year=252)

,Total Return,CAR,Sharpe,Max Drawdown,Trades
0,-0.043867,-0.05576,-2.380129,-0.062338,58
